# llm_reports.ipynb
## Generación automática de informes clínicos mediante LLM
### TFG — Análisis de biomarcadores acústicos en ELA

**Autor:** Jakub Wysocki  

---

Este notebook implementa el módulo de generación automática de informes clínicos
a partir de los resultados del pipeline de ML. Utiliza la API de Groq con el modelo
Llama-3.3-70B (free tier).

**Los tres tipos de informe generados:**

| Tipo | Descripción | Destinatario |
|------|-------------|-------------|
| Estado acústico | Síntesis descriptiva de biomarcadores de la sesión | Clínico / Investigador |
| Clasificación | Output del modelo + SHAP contextualizado | Clínico |
| Evolutivo | Comparativa longitudinal entre sesiones | Clínico / Seguimiento |

**Stack:** Groq API · Llama-3.3-70B-Versatile · python-dotenv  
**Privacidad:** El LLM nunca recibe datos identificativos del paciente, solo métricas numéricas anonimizadas.

**Prerequisito:** variable de entorno `GROQ_API_KEY` configurada.

## 0. Importaciones y configuración

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from groq import Groq
from dotenv import load_dotenv

# Cargar variables de entorno desde .env (si existe)
load_dotenv(Path('..').resolve() / '.env')

PROJECT_ROOT  = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR   = PROJECT_ROOT / 'reports' / 'clinical'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Verificar API key ────────────────────────────────────────────────────────
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
if not GROQ_API_KEY:
    raise EnvironmentError(
        'GROQ_API_KEY no encontrada.\n'
        'Configúrala antes de ejecutar este notebook:\n'
        '  Windows PowerShell: $env:GROQ_API_KEY = "tu_clave"\n'
        '  O añádela al archivo .env en la raíz del proyecto.'
    )

client = Groq(api_key=GROQ_API_KEY)
MODEL  = 'llama-3.3-70b-versatile'

print(f'✓ API key cargada correctamente')
print(f'  Modelo : {MODEL}')
print(f'  Informes → {REPORTS_DIR}')

## 1. Carga de datos de entrada

In [ ]:
# Dataset final
df = pd.read_csv(PROCESSED_DIR / 'dataset_final.csv')
META_COLS    = ['subject_id', 'genero', 'label_clinico', 'label_maquina']
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]
VOCALS       = ['a', 'e', 'i', 'o', 'u']

# Resultados de ML
results_df = pd.read_csv(PROCESSED_DIR / 'ml_results_base.csv')

# SHAP importance
shap_df = pd.read_csv(PROCESSED_DIR / 'shap_importance.csv')

print(f'Dataset      : {df.shape}  ({len(df)} sujetos × {len(FEATURE_COLS)} features)')
print(f'Resultados ML: {results_df.shape}')
print(f'SHAP features: {len(shap_df)}  (top-5: {shap_df["feature"].head().tolist()})')

## 2. Módulo de generación LLM

Funciones centrales del pipeline de generación de informes.

In [ ]:
def call_groq(
    prompt: str,
    system_prompt: str,
    max_tokens: int = 1200,
    temperature: float = 0.3,
) -> str:
    """
    Realiza una llamada a la API de Groq y devuelve el texto generado.

    Parameters
    ----------
    prompt : str
        Prompt de usuario con los datos estructurados del paciente.
    system_prompt : str
        Instrucciones de rol y formato para el modelo.
    max_tokens : int
        Longitud máxima del informe generado.
    temperature : float
        Temperatura de muestreo. Valor bajo (0.3) para mayor fidelidad
        a los datos de entrada y reducción de alucinaciones.

    Returns
    -------
    str
        Texto del informe generado.
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system',  'content': system_prompt},
            {'role': 'user',    'content': prompt},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()


def get_patient_features(
    df: pd.DataFrame,
    subject_id: str,
    top_shap_features: list[str],
) -> dict:
    """
    Extrae los valores de los biomarcadores más relevantes (SHAP) para
    un sujeto concreto, junto con sus metadatos anonimizados.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset final.
    subject_id : str
        Identificador del sujeto.
    top_shap_features : list[str]
        Lista de features ordenadas por importancia SHAP.

    Returns
    -------
    dict
        Metadatos anonimizados + valores de biomarcadores.
    """
    row = df[df['subject_id'] == subject_id].iloc[0]

    genero_str = 'hombre' if row['genero'] == 1 else 'mujer'

    # Top-10 features SHAP con sus valores
    top_features = {
        feat: round(float(row[feat]), 4)
        for feat in top_shap_features[:10]
        if feat in row.index
    }

    # Estadísticas por vocal (media de features de esa vocal)
    vocal_summaries = {}
    for vocal in VOCALS:
        cols = [c for c in FEATURE_COLS if c.endswith(f'_{vocal}')]
        if cols:
            vocal_summaries[vocal] = {
                'mean': round(float(row[cols].mean()), 4),
                'std':  round(float(row[cols].std()),  4),
            }

    return {
        'genero':          genero_str,
        'label_clinico':   row['label_clinico'],
        'label_maquina':   row['label_maquina'],
        'top_shap_values': top_features,
        'vocal_summaries': vocal_summaries,
    }


def get_best_model_result(
    results_df: pd.DataFrame,
    label_system: str = 'clinico',
) -> dict:
    """
    Obtiene el resultado del mejor modelo global para el sistema de
    etiquetado indicado.
    """
    subset = results_df[results_df['label_system'] == label_system]
    best   = subset.loc[subset['auc_roc_mean'].idxmax()]
    return {
        'modelo':      best['model'],
        'escenario':   best['scenario'],
        'auc_roc':     round(float(best['auc_roc_mean']), 3),
        'auc_std':     round(float(best['auc_roc_std']),  3),
        'f1_macro':    round(float(best['f1_macro_mean']), 3),
        'recall':      round(float(best['recall_mean']),   3),
        'specificity': round(float(best['specificity_mean']), 3),
    }


SYSTEM_PROMPT_BASE = """Eres un asistente especializado en el análisis de biomarcadores acústicos \
para la Esclerosis Lateral Amiotrófica (ELA). Generas informes clínicos estructurados a partir de \
datos numéricos de análisis de voz y resultados de modelos de clasificación. \
Tus informes son claros, precisos y están orientados al apoyo a la interpretación clínica. \
IMPORTANTE: Siempre incluyes al final del informe un aviso explícito de que el informe ha sido \
generado automáticamente y requiere supervisión de un profesional sanitario cualificado. \
No realizas diagnósticos. No afirmas nada que no esté respaldado por los datos proporcionados."""

print('✓ Funciones del módulo LLM definidas')

## 3. Informe tipo 1 — Estado acústico

Síntesis descriptiva de los biomarcadores acústicos de la sesión actual.
Proporciona contexto sobre qué mide cada biomarcador y cómo se interpreta
en el contexto de la ELA.

In [ ]:
def build_acoustic_report_prompt(
    patient_data: dict,
    top_shap_features: list[str],
    population_stats: dict,
) -> str:
    """
    Construye el prompt para el informe de estado acústico.

    Incluye los valores del paciente, las medias poblacionales de referencia
    y las features más discriminativas según SHAP.
    """
    vocal_lines = '\n'.join([
        f"  - Vocal '{v}': media de features = {s['mean']:.4f}, desv.típica = {s['std']:.4f}"
        for v, s in patient_data['vocal_summaries'].items()
    ])

    shap_lines = '\n'.join([
        f"  - {feat}: {val:.4f}  (referencia poblacional: {population_stats.get(feat, {}).get('mean', 'N/D'):.4f})"
        for feat, val in patient_data['top_shap_values'].items()
    ])

    return f"""Genera un INFORME DE ESTADO ACÚSTICO para un paciente con ELA.

DATOS DEL PACIENTE (anonimizado):
  Sexo biológico: {patient_data['genero']}
  Diagnóstico clínico: {patient_data['label_clinico']}

PERFIL ACÚSTICO POR VOCAL:
{vocal_lines}

BIOMARCADORES MÁS DISCRIMINATIVOS (según análisis SHAP):
{shap_lines}

INSTRUCCIONES DE FORMATO:
1. Comienza con un resumen ejecutivo (2-3 frases).
2. Sección 'Análisis por vocal': interpreta brevemente el perfil de cada vocal.
3. Sección 'Biomarcadores destacados': explica qué significa clínicamente cada uno de los 5 biomarcadores más importantes y si el valor del paciente sugiere alteración.
4. Sección 'Interpretación global': síntesis del perfil acústico del paciente en el contexto de la ELA.
5. Aviso obligatorio de supervisión profesional.
Usa lenguaje técnico pero comprensible para un neurólogo o logopeda.
NO hagas afirmaciones diagnósticas definitivas."""


def generate_acoustic_report(
    subject_id: str,
    df: pd.DataFrame,
    shap_df: pd.DataFrame,
) -> str:
    """Genera y devuelve el informe de estado acústico para un sujeto."""
    top_features = shap_df['feature'].tolist()
    patient_data = get_patient_features(df, subject_id, top_features)

    # Estadísticas de referencia poblacional (media del grupo de su clase)
    group_df = df[df['label_clinico'] == patient_data['label_clinico']]
    population_stats = {
        feat: {'mean': round(float(group_df[feat].mean()), 4)}
        for feat in top_features[:10]
        if feat in group_df.columns
    }

    prompt = build_acoustic_report_prompt(patient_data, top_features, population_stats)
    return call_groq(prompt, SYSTEM_PROMPT_BASE, max_tokens=1200)


print('✓ Generador de informe acústico definido')

In [ ]:
# Generar informe acústico para un sujeto de ejemplo
SUBJECT_EXAMPLE = df[df['label_clinico'] == 'ELA_bulbar']['subject_id'].iloc[0]
print(f'Generando informe acústico para: {SUBJECT_EXAMPLE}')
print('=' * 65)

report_acoustic = generate_acoustic_report(SUBJECT_EXAMPLE, df, shap_df)
print(report_acoustic)

# Guardar
out_path = REPORTS_DIR / f'informe_acustico_{SUBJECT_EXAMPLE}.txt'
out_path.write_text(report_acoustic, encoding='utf-8')
print(f'\n✓ Guardado: {out_path}')

## 4. Informe tipo 2 — Clasificación

Explicación contextualizada del output del modelo predictivo.
Incluye la clase predicha, la probabilidad, el rendimiento del modelo
y una interpretación de los biomarcadores SHAP más influyentes.

In [ ]:
def build_classification_report_prompt(
    patient_data: dict,
    model_result: dict,
    shap_top5: list[tuple],
    label_system: str,
) -> str:
    """
    Construye el prompt para el informe de clasificación.

    Parameters
    ----------
    patient_data : dict
        Datos del paciente extraídos por get_patient_features.
    model_result : dict
        Métricas del mejor modelo.
    shap_top5 : list[tuple]
        Lista de (feature, shap_value) de los 5 biomarcadores más influyentes.
    label_system : str
        'clinico' o 'maquina'.
    """
    label_key  = 'label_clinico' if label_system == 'clinico' else 'label_maquina'
    clase_pred = patient_data[label_key]
    sistema_str = 'etiquetado clínico convencional' if label_system == 'clinico' \
                  else 'reetiquetado computacional (S4VM)'

    shap_lines = '\n'.join([
        f"  {i+1}. {feat}: importancia SHAP = {val:.5f}"
        for i, (feat, val) in enumerate(shap_top5)
    ])

    return f"""Genera un INFORME DE CLASIFICACIÓN para un paciente con ELA.

RESULTADO DEL MODELO ({sistema_str}):
  Clase asignada al paciente: {clase_pred}
  Escenario de clasificación: {model_result['escenario']}
  Modelo utilizado: {model_result['modelo']}
  Rendimiento del modelo (validación cruzada k=5):
    - AUC-ROC      : {model_result['auc_roc']} ± {model_result['auc_std']}
    - F1-macro     : {model_result['f1_macro']}
    - Sensibilidad : {model_result['recall']}
    - Especificidad: {model_result['specificity']}

FACTORES ACÚSTICOS MÁS INFLUYENTES EN LA CLASIFICACIÓN (análisis SHAP):
{shap_lines}

DATOS DEL PACIENTE:
  Sexo biológico: {patient_data['genero']}

INSTRUCCIONES DE FORMATO:
1. Resumen ejecutivo: clase asignada y confianza del modelo (2-3 frases).
2. Sección 'Rendimiento del modelo': interpreta las métricas en contexto clínico. Explica qué significa AUC-ROC={model_result['auc_roc']} para un clínico.
3. Sección 'Factores determinantes': explica qué mide cada uno de los 5 biomarcadores SHAP y por qué son relevantes en este caso.
4. Sección 'Limitaciones e interpretación': señala las limitaciones del modelo (n=63, desbalanceo de clases, ausencia de significancia individual en Kruskal-Wallis) y cómo debe interpretarse el resultado.
5. Aviso obligatorio de supervisión profesional.
Tono técnico pero comprensible para un neurólogo."""


def generate_classification_report(
    subject_id: str,
    df: pd.DataFrame,
    results_df: pd.DataFrame,
    shap_df: pd.DataFrame,
    label_system: str = 'clinico',
) -> str:
    """Genera y devuelve el informe de clasificación para un sujeto."""
    top_features = shap_df['feature'].tolist()
    patient_data = get_patient_features(df, subject_id, top_features)
    model_result = get_best_model_result(results_df, label_system)

    shap_top5 = [
        (row['feature'], row['mean_shap'])
        for _, row in shap_df.head(5).iterrows()
    ]

    prompt = build_classification_report_prompt(
        patient_data, model_result, shap_top5, label_system
    )
    return call_groq(prompt, SYSTEM_PROMPT_BASE, max_tokens=1300)


print('✓ Generador de informe de clasificación definido')

In [ ]:
# Generar informe de clasificación — etiquetado clínico
print(f'Generando informe de clasificación (clínico) para: {SUBJECT_EXAMPLE}')
print('=' * 65)

report_classification = generate_classification_report(
    SUBJECT_EXAMPLE, df, results_df, shap_df, label_system='clinico'
)
print(report_classification)

out_path = REPORTS_DIR / f'informe_clasificacion_{SUBJECT_EXAMPLE}_clinico.txt'
out_path.write_text(report_classification, encoding='utf-8')
print(f'\n✓ Guardado: {out_path}')

In [ ]:
# Generar informe de clasificación — reetiquetado máquina
# Relevante para pacientes cuya etiqueta cambia entre sistemas
diff_subjects = df[df['label_clinico'] != df['label_maquina']]['subject_id']

if len(diff_subjects) > 0:
    subject_reetiq = diff_subjects.iloc[0]
    print(f'Generando informe de clasificación (máquina) para sujeto reetiquetado: {subject_reetiq}')
    row_info = df[df['subject_id'] == subject_reetiq].iloc[0]
    print(f'  label_clinico: {row_info["label_clinico"]}  →  label_maquina: {row_info["label_maquina"]}')
    print('=' * 65)

    report_maquina = generate_classification_report(
        subject_reetiq, df, results_df, shap_df, label_system='maquina'
    )
    print(report_maquina)

    out_path = REPORTS_DIR / f'informe_clasificacion_{subject_reetiq}_maquina.txt'
    out_path.write_text(report_maquina, encoding='utf-8')
    print(f'\n✓ Guardado: {out_path}')
else:
    print('No hay sujetos reetiquetados en el dataset.')

## 5. Informe tipo 3 — Evolutivo (longitudinal)

Resumen comparativo de la evolución de los biomarcadores entre dos sesiones.
En el contexto de este trabajo, simula dos sesiones a partir de
perturbaciones controladas del sujeto original, ilustrando la capacidad
del sistema para detectar cambios longitudinales.

In [ ]:
def simulate_longitudinal_session(
    row: pd.Series,
    feature_cols: list[str],
    noise_level: float = 0.05,
    seed: int = 42,
) -> pd.Series:
    """
    Simula una sesión de seguimiento posterior añadiendo ruido gaussiano
    controlado a los biomarcadores de la sesión base.

    En un sistema real, esta función sería reemplazada por la carga de
    los datos de la sesión real del paciente.

    Parameters
    ----------
    row : pd.Series
        Fila del dataset (sesión base).
    feature_cols : list[str]
        Columnas de features acústicas.
    noise_level : float
        Fracción de la desviación estándar del dataset a añadir como ruido.
    seed : int
        Semilla para reproducibilidad.
    """
    rng = np.random.default_rng(seed)
    row_sim = row.copy()
    for feat in feature_cols:
        std = df[feat].std()
        row_sim[feat] = row[feat] + rng.normal(0, noise_level * std)
    return row_sim


def build_evolutionary_report_prompt(
    patient_data_s1: dict,
    patient_data_s2: dict,
    session_dates: tuple[str, str],
    shap_top5: list[str],
) -> str:
    """
    Construye el prompt para el informe evolutivo.

    Compara los biomarcadores SHAP más relevantes entre dos sesiones.
    """
    changes = []
    for feat in shap_top5:
        v1 = patient_data_s1['top_shap_values'].get(feat, 0)
        v2 = patient_data_s2['top_shap_values'].get(feat, 0)
        delta = v2 - v1
        pct   = (delta / abs(v1) * 100) if v1 != 0 else 0
        direction = 'aumentó' if delta > 0 else 'disminuyó'
        changes.append(
            f"  - {feat}: {v1:.4f} → {v2:.4f}  ({direction} un {abs(pct):.1f}%)"
        )

    vocal_changes = []
    for vocal in VOCALS:
        m1 = patient_data_s1['vocal_summaries'].get(vocal, {}).get('mean', 0)
        m2 = patient_data_s2['vocal_summaries'].get(vocal, {}).get('mean', 0)
        delta = m2 - m1
        direction = 'aumentó' if delta > 0 else 'disminuyó'
        vocal_changes.append(f"  - Vocal '{vocal}': {m1:.4f} → {m2:.4f}  ({direction})")

    return f"""Genera un INFORME EVOLUTIVO (LONGITUDINAL) para un paciente con ELA.

DATOS DEL PACIENTE (anonimizado):
  Sexo biológico: {patient_data_s1['genero']}
  Diagnóstico clínico: {patient_data_s1['label_clinico']}

COMPARATIVA DE BIOMARCADORES DISCRIMINATIVOS (SHAP):
  Sesión 1 ({session_dates[0]}) → Sesión 2 ({session_dates[1]}):
{'chr(10)'.join(changes)}

PERFIL VOCAL COMPARATIVO:
{'chr(10)'.join(vocal_changes)}

INSTRUCCIONES DE FORMATO:
1. Resumen ejecutivo: tendencia general observada en 2-3 frases.
2. Sección 'Evolución de biomarcadores': interpreta los cambios en cada biomarcador clave y su significado clínico en ELA.
3. Sección 'Perfil vocal': comenta si la evolución es consistente entre vocales o si alguna vocal muestra mayor deterioro.
4. Sección 'Tendencia global': sintetiza si los datos sugieren estabilidad, progresión leve o deterioro significativo.
5. Aviso obligatorio de supervisión profesional.
Sé específico con los cambios cuantitativos. No specules sobre causas no respaldadas por los datos."""


def generate_evolutionary_report(
    subject_id: str,
    df: pd.DataFrame,
    shap_df: pd.DataFrame,
    noise_level: float = 0.05,
) -> str:
    """Genera el informe evolutivo comparando dos sesiones del mismo sujeto."""
    top_features = shap_df['feature'].tolist()

    # Sesión 1: datos reales
    patient_data_s1 = get_patient_features(df, subject_id, top_features)

    # Sesión 2: simulación de seguimiento (en un sistema real: carga de nuevos datos)
    row_s1 = df[df['subject_id'] == subject_id].iloc[0]
    row_s2 = simulate_longitudinal_session(row_s1, FEATURE_COLS, noise_level)

    # Construir patient_data_s2 manualmente
    patient_data_s2 = {
        'genero':          patient_data_s1['genero'],
        'label_clinico':   patient_data_s1['label_clinico'],
        'label_maquina':   patient_data_s1['label_maquina'],
        'top_shap_values': {
            feat: round(float(row_s2[feat]), 4)
            for feat in top_features[:10]
            if feat in row_s2.index
        },
        'vocal_summaries': {
            vocal: {
                'mean': round(float(row_s2[[c for c in FEATURE_COLS if c.endswith(f'_{vocal}')]].mean()), 4),
                'std':  round(float(row_s2[[c for c in FEATURE_COLS if c.endswith(f'_{vocal}')]].std()),  4),
            }
            for vocal in VOCALS
        },
    }

    # Fechas ficticias para ilustrar el formato
    now = datetime.now()
    date_s2 = now.strftime('%d/%m/%Y')
    date_s1 = now.replace(month=max(1, now.month - 3)).strftime('%d/%m/%Y')

    prompt = build_evolutionary_report_prompt(
        patient_data_s1, patient_data_s2,
        session_dates=(date_s1, date_s2),
        shap_top5=top_features[:5],
    )
    return call_groq(prompt, SYSTEM_PROMPT_BASE, max_tokens=1400)


print('✓ Generador de informe evolutivo definido')

In [ ]:
print(f'Generando informe evolutivo para: {SUBJECT_EXAMPLE}')
print('=' * 65)

report_evolutionary = generate_evolutionary_report(SUBJECT_EXAMPLE, df, shap_df)
print(report_evolutionary)

out_path = REPORTS_DIR / f'informe_evolutivo_{SUBJECT_EXAMPLE}.txt'
out_path.write_text(report_evolutionary, encoding='utf-8')
print(f'\n✓ Guardado: {out_path}')

## 6. Generación por lotes — todos los sujetos ELA_bulbar

Genera los tres tipos de informe para todos los sujetos con ELA_bulbar
(etiquetado clínico). Útil para evaluar la coherencia del sistema a escala.

In [ ]:
def generate_all_reports(
    df: pd.DataFrame,
    results_df: pd.DataFrame,
    shap_df: pd.DataFrame,
    target_label: str = 'ELA_bulbar',
    label_col: str = 'label_clinico',
    sleep_between: float = 1.5,
) -> pd.DataFrame:
    """
    Genera los tres tipos de informe para todos los sujetos de la clase
    indicada y guarda un registro de metadatos.

    Parameters
    ----------
    target_label : str
        Clase objetivo ('ELA_bulbar', 'ELA_no_bulbar', 'Control').
    sleep_between : float
        Segundos de pausa entre llamadas (respeto al rate limit de Groq).

    Returns
    -------
    pd.DataFrame
        Registro de informes generados con metadatos.
    """
    subjects = df[df[label_col] == target_label]['subject_id'].tolist()
    print(f'Generando informes para {len(subjects)} sujetos con {target_label}...')

    records = []
    for i, sid in enumerate(subjects, 1):
        print(f'[{i:02d}/{len(subjects)}] {sid}', end=' ')
        record = {'subject_id': sid, 'label': target_label, 'timestamp': datetime.now().isoformat()}

        for report_type, func, kwargs in [
            ('acustico',       generate_acoustic_report,        {'df': df, 'shap_df': shap_df}),
            ('clasificacion',  generate_classification_report,  {'df': df, 'results_df': results_df, 'shap_df': shap_df}),
            ('evolutivo',      generate_evolutionary_report,    {'df': df, 'shap_df': shap_df}),
        ]:
            try:
                text = func(sid, **kwargs)
                path = REPORTS_DIR / f'informe_{report_type}_{sid}.txt'
                path.write_text(text, encoding='utf-8')
                record[f'{report_type}_ok'] = True
                print(f'{report_type}✓', end=' ')
                time.sleep(sleep_between)
            except Exception as e:
                record[f'{report_type}_ok'] = False
                record[f'{report_type}_error'] = str(e)
                print(f'{report_type}✗', end=' ')

        records.append(record)
        print()

    log_df = pd.DataFrame(records)
    log_df.to_csv(PROCESSED_DIR / f'report_log_{target_label}.csv', index=False)
    print(f'\n✓ Registro guardado: report_log_{target_label}.csv')
    return log_df


print('✓ Generador por lotes definido')
print('  Para ejecutar: log = generate_all_reports(df, results_df, shap_df)')

In [ ]:
# Ejecutar generación por lotes para ELA_bulbar
# (descomenta la línea siguiente cuando quieras generar todos los informes)
log_df = generate_all_reports(df, results_df, shap_df, target_label='ELA_bulbar')

print('Generación por lotes no ejecutada automáticamente.')
print('Descomenta la línea de arriba cuando quieras generar todos los informes.')

## 7. Evaluación de coherencia de los informes

Verificación automática de que los valores numéricos referenciados
en el informe generado son consistentes con los datos de entrada.

In [ ]:
def evaluate_report_coherence(
    report_text: str,
    expected_values: dict,
    tolerance: float = 0.05,
) -> dict:
    """
    Verifica que los valores numéricos en el informe son coherentes con
    los datos de entrada. Mitiga el riesgo de alucinaciones factuales.

    Parameters
    ----------
    report_text : str
        Texto del informe generado por el LLM.
    expected_values : dict
        Diccionario {valor_esperado: descripcion} de los valores clave.
    tolerance : float
        Tolerancia relativa para comparaciones numéricas.

    Returns
    -------
    dict
        Informe de coherencia con 'total', 'passed', 'failed', 'details'.
    """
    import re

    # Extraer todos los números del informe
    numbers_in_report = set(
        float(n) for n in re.findall(r'-?\d+\.\d+|-?\d+', report_text)
    )

    results = {'total': len(expected_values), 'passed': 0, 'failed': 0, 'details': []}

    for expected_val, description in expected_values.items():
        # Buscar si algún número en el informe está dentro de la tolerancia
        found = any(
            abs(n - expected_val) <= tolerance * abs(expected_val) + 0.001
            for n in numbers_in_report
        )
        status = 'PASS' if found else 'WARN'
        if found:
            results['passed'] += 1
        else:
            results['failed'] += 1
        results['details'].append(f'  [{status}] {description}: {expected_val}')

    return results


# Evaluar coherencia del informe de clasificación
best_model = get_best_model_result(results_df, 'clinico')
expected = {
    best_model['auc_roc']:     f'AUC-ROC del mejor modelo ({best_model["modelo"]})',
    best_model['f1_macro']:    'F1-macro del mejor modelo',
    best_model['recall']:      'Sensibilidad del mejor modelo',
    best_model['specificity']: 'Especificidad del mejor modelo',
}

coherence = evaluate_report_coherence(report_classification, expected)

print('=== EVALUACIÓN DE COHERENCIA — Informe de clasificación ===')
print(f'Valores verificados : {coherence["total"]}')
print(f'Coherentes (PASS)   : {coherence["passed"]}')
print(f'No encontrados (WARN): {coherence["failed"]}  '
      '(puede ser que el LLM los reformuló o redondeó)')
print('Detalle:')
for d in coherence['details']:
    print(d)

## 8. Resumen final

In [ ]:
print('=' * 65)
print('RESUMEN DEL MÓDULO DE GENERACIÓN DE INFORMES LLM')
print('=' * 65)

print(f'\n[Configuración]')
print(f'  Modelo          : {MODEL}')
print(f'  Proveedor       : Groq API (free tier)')
print(f'  Temperatura     : 0.3 (baja → mayor fidelidad a los datos)')
print(f'  Sujetos en dataset: {len(df)}')

print(f'\n[Tipos de informe implementados]')
print(f'  1. Informe de estado acústico    → snapshot biomarcadores sesión')
print(f'  2. Informe de clasificación      → output modelo + SHAP contextualizado')
print(f'  3. Informe evolutivo             → comparativa longitudinal entre sesiones')

print(f'\n[Informes generados en esta ejecución]')
reports = list(REPORTS_DIR.glob('*.txt'))
for r in sorted(reports):
    size = r.stat().st_size
    print(f'  {r.name}  ({size:,} bytes)')

print(f'\n[Evaluación de coherencia]')
print(f'  Valores numéricos verificados: {coherence["total"]}')
print(f'  Coherentes: {coherence["passed"]} / {coherence["total"]}')

print(f'\n[Garantías del sistema]')
print(f'  ✓ Datos anonimizados en todos los prompts')
print(f'  ✓ Temperatura baja (0.3) para reducir alucinaciones')
print(f'  ✓ Aviso de supervisión profesional en todos los informes')
print(f'  ✓ Verificación automática de coherencia numérica')

print('\n' + '=' * 65)
print('→ Pipeline completo finalizado.')
print('  Siguiente: API + interfaz web (llm_api.py / frontend)')
print('=' * 65)

---
## Checklist para la memoria (Sección 3.6 y Capítulo 4)

### Sección 3.6 — Metodología LLM
| Elemento | Implementación |
|----------|----------------|
| Stack: Groq + Llama-3.3-70B | `call_groq()` con MODEL |
| Generación condicionada | Prompt estructurado con datos numéricos |
| 3 tipos de informe | `generate_acoustic/classification/evolutionary_report()` |
| Temperatura 0.3 | Reducción de alucinaciones |
| Anonimización | Sin datos identificativos en el prompt |
| Evaluación de coherencia | `evaluate_report_coherence()` |
| Aviso supervisión | Incluido en `SYSTEM_PROMPT_BASE` |

### Capítulo 4 — Resultados LLM
Incluir en la memoria:
- Ejemplo de los 3 tipos de informe generados (adjuntar como apéndice o extracto)
- Resultado de la evaluación de coherencia numérica
- Análisis cualitativo: claridad, adecuación clínica, ausencia de afirmaciones no fundamentadas

**Pipeline completo finalizado:**
```
MATLAB (extracción) → preprocessing.py → label_loader.py
→ label_integration.ipynb → eda.ipynb → ml_experiments.ipynb
→ llm_reports.ipynb  ← estás aquí
→ API + interfaz web  (siguiente paso)
```